# Question 12 — Rosenbrock Function: Gradient Descent vs Newton (Detailed)

This notebook implements and compares **Steepest Descent (Gradient Descent)** and **Newton's method** on the Rosenbrock function, with expanded derivations and theoretical background.

Contents:
- Problem statement and function definition
- Analytic derivation of gradient and Hessian (step-by-step)
- Algorithm descriptions (GD, Backtracking/Armijo, Newton), pseudocode and formulas
- Convergence concepts: linear vs quadratic, Taylor-based derivation for Newton
- Implementations and experiments (runs from \(x_0=[-1.2,1]^\top\))
- Plots: trajectories, convergence (\(\log_{10} f\)), gradient norms
- Thorough discussion and practical recommendations


## Problem statement

We consider the Rosenbrock function
$$[f(x_1,x_2)=100(x_2 - x_1^2)^2 + (1-x_1)^2,]$$
with global minimizer
$$ (x^*=[1,1]^top) and (f(x^*)=0).$$

Goal: compare optimization methods with respect to iterations, convergence speed and robustness.

## (a) Analytic derivatives — gradient and Hessian (full derivation)

Define residual \(r(x)=x_2 - x_1^2\). Then
\(f(x)=100 r(x)^2 + (1-x_1)^2\).

### Gradient (first derivatives)
Using the chain rule:
\begin{align*}
\frac{\partial f}{\partial x_1} &= 200 r \cdot \frac{\partial r}{\partial x_1} - 2(1-x_1) \\
&= 200 (x_2 - x_1^2)(-2 x_1) - 2(1-x_1) \\
&= -400 x_1 (x_2 - x_1^2) - 2(1-x_1).
\end{align*}
\[
\frac{\partial f}{\partial x_2} = 200 (x_2 - x_1^2).
\]
So in vector form:
\[
\nabla f(x)=\begin{bmatrix}-400 x_1 (x_2 - x_1^2) - 2(1-x_1) \\[6pt] 200(x_2 - x_1^2)\end{bmatrix}.
\]

### Hessian (second derivatives)
Differentiate gradient entries:
\begin{align*}
\frac{\partial^2 f}{\partial x_1^2} &= -400\big[(x_2 - x_1^2) + x_1(-2x_1)\big] + 2 \\
&= -400(x_2 - x_1^2) + 800 x_1^2 + 2 \\
&= 1200 x_1^2 - 400 x_2 + 2.
\end{align*}
Mixed derivative:
\[
\frac{\partial^2 f}{\partial x_1\partial x_2} = -400 x_1,
\]
and
\[
\frac{\partial^2 f}{\partial x_2^2} = 200.
\]
So
\[
H(x)=\begin{bmatrix}1200 x_1^2 - 400 x_2 + 2 & -400 x_1 \\ -400 x_1 & 200\end{bmatrix}.
\]

These expressions are used in the code below.

## Convergence concepts — linear vs quadratic (brief theory)

- **Linear convergence**: error sequence \(e_k=\|x_k-x^*\|\) satisfies $(e_{k+1}\le C e_k)$ for some $(0<C<1)$. GD often displays linear convergence near a strongly convex minimum, rate depends on condition number.
- **Quadratic convergence**: $(e_{k+1}\le C e_k^2)$ when close enough to solution. Newton's method (with exact Hessian) attains quadratic convergence under standard regularity assumptions.

### Taylor derivation (why Newton is quadratic)
Let $(x^*)$ be a minimizer with $(\nabla f(x^*)=0)$ and nonsingular Hessian $(H^*=H(x^*))$. Expand gradient at $(x_k)$:
$$[
\nabla f(x_k) = H^*(x_k-x^*) + R(x_k),
]$$
where the remainder $(\|R(x_k)\|=O(\|x_k-x^*\|^2))$. Newton step solves $(H(x_k)p_k=-\nabla f(x_k))$ so roughly
$(p_k \approx -(H^*)^{-1}H^*(x_k-x^*) + O(\|x_k-x^*\|^2) = -(x_k-x^*) + O(\|x_k-x^*\|^2))$.
Therefore the new error $(x_{k+1}-x^* = O(\|x_k-x^*\|^2))$, i.e. quadratic.

This is the intuitive reason Newton is much faster near optimum.

## Algorithms — pseudocode and Armijo condition

### Steepest Descent (fixed-step)
$[
x_{k+1} = x_k - \alpha \nabla f(x_k)
]$
Choose small $(\alpha)$ for stability; too small = slow, too large = divergence.

### Backtracking (Armijo) line search
Given descent direction $(p_k)$ (e.g. $(-\nabla f)$), find $(\alpha)$ such that
$[
f(x_k + \alpha p_k) \le f(x_k) + c \alpha \nabla f(x_k)^\top p_k,
]$
with typical $(c=10^{-4})$ and shrink factor $(\rho\in(0,1))$ (we try $(\alpha\leftarrow\rho\alpha)$ until condition holds).

### Newton's method
$[
H(x_k) p_k = -\nabla f(x_k),\qquad x_{k+1}=x_k + p_k.
]$
Add damping if $(f(x_k+p_k)>f(x_k))$ (halve the step until satisfied).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
np.set_printoptions(precision=6, suppress=True)

## Implementations: function, gradient, Hessian

In [2]:
def rosenbrock(x):
    x1, x2 = x[0], x[1]
    return 100.0*(x2 - x1**2)**2 + (1.0 - x1)**2

def grad_rosenbrock(x):
    x1, x2 = x[0], x[1]
    df_dx1 = -400.0*x1*(x2 - x1**2) - 2.0*(1.0 - x1)
    df_dx2 = 200.0*(x2 - x1**2)
    return np.array([df_dx1, df_dx2])

def hess_rosenbrock(x):
    x1, x2 = x[0], x[1]
    H11 = 1200.0*x1**2 - 400.0*x2 + 2.0
    H12 = -400.0*x1
    H22 = 200.0
    return np.array([[H11, H12],[H12, H22]])

## Backtracking line search (Armijo condition) implementation

In [3]:
def backtracking_line_search(x, p, f, grad_f, alpha0=1.0, rho=0.5, c=1e-4):
    alpha = alpha0
    fx = f(x)
    g_dot_p = np.dot(grad_f(x), p)
    while True:
        x_new = x + alpha * p
        if f(x_new) <= fx + c * alpha * g_dot_p:
            return alpha
        alpha *= rho
        if alpha < 1e-12:
            return alpha

## Steepest Descent (GD) implementation

In [4]:
def steepest_descent(x0, alpha=0.001, max_iter=50, tol=1e-6, use_backtracking=False):
    x = x0.astype(float).copy()
    traj = [x.copy()]
    fvals = [rosenbrock(x)]
    grad_norms = [np.linalg.norm(grad_rosenbrock(x))]
    for k in range(max_iter):
        g = grad_rosenbrock(x)
        if np.linalg.norm(g) < tol:
            break
        p = -g
        step = backtracking_line_search(x, p, rosenbrock, grad_rosenbrock) if use_backtracking else alpha
        x = x + step * p
        traj.append(x.copy())
        fvals.append(rosenbrock(x))
        grad_norms.append(np.linalg.norm(grad_rosenbrock(x)))
    return np.array(traj), np.array(fvals), np.array(grad_norms)

## Newton's method implementation (with damping safeguard)

In [5]:
def newton_method(x0, max_iter=50, tol=1e-6, damp=True):
    x = x0.astype(float).copy()
    traj = [x.copy()]
    fvals = [rosenbrock(x)]
    grad_norms = [np.linalg.norm(grad_rosenbrock(x))]
    for k in range(max_iter):
        g = grad_rosenbrock(x)
        if np.linalg.norm(g) < tol:
            break
        H = hess_rosenbrock(x)
        try:
            p = -np.linalg.solve(H, g)
        except np.linalg.LinAlgError:
            p = -g
        step = 1.0
        if damp:
            fx = rosenbrock(x)
            while step > 1e-8:
                x_new = x + step * p
                if rosenbrock(x_new) <= fx + 1e-10:
                    break
                step *= 0.5
        x = x + step * p
        traj.append(x.copy())
        fvals.append(rosenbrock(x))
        grad_norms.append(np.linalg.norm(grad_rosenbrock(x)))
    return np.array(traj), np.array(fvals), np.array(grad_norms)

## Plot helpers (contour + trajectories, log f, gradient norms)

In [6]:
def plot_contour_with_trajectories(trajs, labels, levels=None, title='Trajectories on Rosenbrock'):
    x1 = np.linspace(-2.0, 2.0, 400)
    x2 = np.linspace(-1.0, 3.0, 400)
    X1, X2 = np.meshgrid(x1, x2)
    Z = 100.0*(X2 - X1**2)**2 + (1.0 - X1)**2
    plt.figure(figsize=(8,6))
    if levels is None:
        levels = np.logspace(-1, 3.5, 40)
    plt.contour(X1, X2, Z, levels=levels, norm=LogNorm(), cmap='viridis')
    for traj, lab in zip(trajs, labels):
        plt.plot(traj[:,0], traj[:,1], '-o', markersize=3, label=lab)
        plt.scatter(traj[0,0], traj[0,1], marker='s', color='black', s=40)
        plt.scatter(traj[-1,0], traj[-1,1], marker='*', color='red', s=80)
    plt.legend()
    plt.title(title)
    plt.xlabel('$x_1$'); plt.ylabel('$x_2$')
    plt.grid(alpha=0.3)
    plt.xlim([-2,2]); plt.ylim([-1,3])
    plt.show()

def plot_logf(fvals_list, labels, title='log10 f(x_k) vs iteration'):
    plt.figure(figsize=(7,4))
    for fvals, lab in zip(fvals_list, labels):
        eps = 1e-16
        plt.plot(np.log10(np.maximum(fvals, eps)), '-o', label=lab)
    plt.xlabel('Iteration'); plt.ylabel('$\\log_{10}(f(x_k))$')
    plt.title(title)
    plt.grid(True); plt.legend(); plt.show()

## Run experiments from $(x_0=[-1.2,1]^\top)$
We run at most 50 iterations or until \(\|\nabla f\|<10^{-6}\).

In [7]:
x0 = np.array([-1.2, 1.0])
max_iter = 50
tol = 1e-6
traj_gd, f_gd, gnorm_gd = steepest_descent(x0, alpha=0.001, max_iter=max_iter, tol=tol, use_backtracking=False)
traj_gd_bt, f_gd_bt, gnorm_gd_bt = steepest_descent(x0, alpha=0.001, max_iter=max_iter, tol=tol, use_backtracking=True)
traj_newton, f_newton, gnorm_newton = newton_method(x0, max_iter=max_iter, tol=tol, damp=True)

In [8]:
print("GD fixed α=0.001:", len(f_gd)-1, "iters | f=", f_gd[-1], "| x=", traj_gd[-1])
print("GD backtracking:", len(f_gd_bt)-1, "iters | f=", f_gd_bt[-1], "| x=", traj_gd_bt[-1])
print("Newton (damped):     ", len(f_newton)-1, "iters | f=", f_newton[-1], "| x=", traj_newton[-1])

GD fixed α=0.001: 50 iters | f= 3.9649126357952045 | x= [-0.98959835  0.98731179]
GD backtracking: 50 iters | f= 0.004834044590886868 | x= [0.93093168 0.86583622]
Newton (damped):      21 iters | f= 3.743975643139474e-21 | x= [1. 1.]


In [9]:
plot_contour_with_trajectories(
    [traj_gd, traj_gd_bt, traj_newton],
    ['GD α=0.001','GD backtracking','Newton (damped)'],
    title='Trajectories of GD (fixed), GD (backtracking), Newton'
)

In [10]:
plot_logf(
    [f_gd, f_gd_bt, f_newton],
    ['GD α=0.001','GD backtracking','Newton (damped)']
)

In [11]:
plt.figure(figsize=(7,4))
plt.semilogy(gnorm_gd, '-o', label='||grad|| GD fixed')
plt.semilogy(gnorm_gd_bt, '-o', label='||grad|| GD backtracking')
plt.semilogy(gnorm_newton, '-o', label='||grad|| Newton')
plt.xlabel('Iteration'); plt.ylabel('Gradient norm')
plt.title('Gradient Norm Comparison')
plt.grid(True); plt.legend(); plt.show()

## Thorough discussion and interpretation

### Why GD (fixed) is slow
- Rosenbrock has a narrow curved valley. The Hessian there is ill-conditioned: one direction has large curvature, the other small. A fixed small step size must be chosen to stay stable in the steep direction, which makes progress slow in the flat direction.
- Geometrically GD moves roughly orthogonal to level sets following \(-\nabla f\) and therefore often zigzags.

### Why backtracking helps
- Line search lets the algorithm use larger steps where safe, accelerating progress along the valley. Armijo ensures sufficient decrease and stability.

### Why Newton is fast
- Newton uses curvature (Hessian) to rescale directions properly: it accounts for anisotropy and takes steps that are approximately the Newton decrement, which leads to quadratic convergence near the minimizer.

### Practical notes
- Newton requires Hessian and solve cost \(O(n^3)\) for dense problems; in high-dimensions use Quasi-Newton (BFGS/L-BFGS).
- If Hessian is singular or indefinite, damping or regularization (Levenberg–Marquardt) helps.
